# 🧪 Lab Supplement: Raman Spectroscopy

[**Profs. Mike Murphy, Lauren Woods, and Jay Foley** — University of North Carolina at Charlotte](https://chemistry.charlotte.edu/)

---

### 🎯 Objectives
- Demonstrate **symmetry principles** in vibrational spectroscopy  
- Predict **IR** and **Raman-active** vibrational modes for a polyatomic molecule  

---

### 📘 Learning Outcomes
By the end of this lab notebook, you should be able to:

1. Identify the **symmetry** of simple molecules  
2. Differentiate between **Raman** and **Infrared (IR)** active vibrational modes  
3. Use **computational and spectroscopic tools** to identify vibrational modes  
4. Calculate **force constants** using spectral data  

---

> 💡 *Tip:*  
> You can run this notebook interactively in Google Colab using the “Open in Colab” button on your Jupyter Book page.





## ⚗️ Approach

The **FTIR spectrum** for *trans*-dichloroethylene, together with Eqs. (3) and (4), determines three of the four force constants —  
$\($ k_{\mathrm{CH}} $\)$, $\($ k_{\alpha} $, and $ k_{\tau} $ — within the approximations applied.  

The remaining force constant, $k_{\mathrm{CC}} $, requires measurement of the **Raman spectrum**, because the carbon–carbon stretch $ \bar{\nu}_{\mathrm{CC}} $ does **not** change the molecular dipole moment and is therefore **absent in the FTIR spectrum**.

---

### 🧮 Deriving  $ k_{\mathrm{CC}} $

Deriving a simple expression for $ k_{\mathrm{CC}} $ is difficult because the modes $ S_1 $, $ S_2 $, and $ S_3 $ are **coupled** in the secular equation.  
The determinant then involves finding the roots of a **cubic polynomial**.

We could use Python to solve the $ 3 \times 3$  determinant for the *trans* isomer,  
but for the *cis* isomer, the system is more complex.  
Since entering and solving the **complete secular equation** (Eq. A10) in Python is not much more difficult,  
we can obtain solutions for **both isomers** by assuming that the bond force constants are the same for each.

Solving the general secular equation thus provides:
- a **check of the validity** of the approach, and  
- insight into the **effect of the approximations** involved.

---

### 💻 Computational Setup

Several scientific programs can perform these calculations,  
but here we’ll use a **Jupyter notebook** built in **Python** to calculate elements of the **G matrix**.

1. A notebook link is provided on Canvas — open it in a browser.  
2. Click the **🚀 Rocket** icon (top right) and select **“Open in Colab.”**  
3. Go to **File → Save a copy in Drive**, and rename it to: `your_last_name_RamanLab`
4. Run the **first code block** to import all required **functions and constants**.  
When successful, you’ll see a ✅ **green checkmark** next to the cell.

---

> 💡 *Tip:*  
> Remember that FTIR and Raman spectroscopy are **complementary techniques** —  
> together, they allow you to determine all four force constants for the molecule.




## 🧰 Importing Required Libraries

In this notebook, we’ll use a few standard Python libraries:

- **NumPy** — for numerical calculations and array handling  
- **SciPy** — for scientific utilities and signal processing  
- **Matplotlib** — for plotting and data visualization  

Run the following cell to import the required modules.


In [ ]:
import numpy as np
from scipy import signal
from matplotlib import pyplot as plt

## 🧾 Preparation

You’ll need your **data collected from Week 1** —  
you *did* bring your lab notebook, right? 😄  

In the next cell, enter the appropriate values after each colon (`:`) for:

- **Bond lengths** (in Å)  
- **Bond angles** (in radians)  
- **Atomic masses** (in amu)  
- **Force constants**

Be sure to record your values carefully —  
these will be used in subsequent calculations of the **G matrix** and **force constants**.


In [ ]:
# STUDENT INPUT SECTION
# Replace placeholder values with experimental or calculated ones as directed in lab handout.

student_input = {
    # Geometry (distances in Å, angles in radians)
    "tau": np.pi,                     # 0 for cis, π for trans
    "C-C bondlength": 1.31,
    "C-H bondlength": 1.07,
    "C-C-H bond angle": 2 * np.pi / 3,
    "C mass": 12.01,
    "H mass": 1.008,

    # Force constants
    "C-C force constant": 7.70,        # kCC  (N/cm)
    "C-H force constant": 5.23,        # kCH  (N/cm)
    "C-C–C-H interaction": 0.00,       # kCCCH (N/cm)
    "C-C–bend interaction": 0.00,      # kCCth (Å N/cm)
    "C-H–bend interaction": 0.00,      # kCHatha (Å N/cm)
    "bend-bend coupling": 0.00,        # kthathb (Å² N/cm)
    "bend force constant": 0.904,      # ktheta (Å² N/cm)
    "torsion force constant": 0.190    # ktau (Å N/cm)
}



## ⚙️ Pre-Written Block: Variable Assignment

The following code cell will automatically **assign variables** for the subsequent 
calculations based on the values you entered in the `student_input` dictionary above.  

➡️ **Do not edit this cell** — just run it once after updating your inputs.

This block:
- extracts bond lengths, angles, and masses,  
- computes inverse quantities (e.g., $1/r_{\mathrm{CC}}$, $1/m_{\mathrm{C}}$), and  
- loads your force constants for use in the $F$ and $G$ matrix calculations that follow.


In [ ]:
# ⚠️ DO NOT MODIFY THIS CELL — JUST RUN IT ONCE AFTER ENTERING YOUR INPUTS

# Geometric parameters
tau = student_input["tau"]
alpha = student_input["C-C-H bond angle"]
rhoCC = 1 / student_input["C-C bondlength"]
rhoCH = 1 / student_input["C-H bondlength"]
muC = 1 / student_input["C mass"]
muH = 1 / student_input["H mass"]

# Force constants
kCC   = student_input["C-C force constant"]
kCH   = student_input["C-H force constant"]
kCCCH = student_input["C-C–C-H interaction"]
kCCth = student_input["C-C–bend interaction"]
kCHatha = student_input["C-H–bend interaction"]
kthathb = student_input["bend-bend coupling"]
ktheta  = student_input["bend force constant"]
ktau    = student_input["torsion force constant"]


## 🧠 Understanding the Force Constants

Before constructing the **F matrix**, let’s review the meaning of each force constant.  
Each constant describes how strongly the potential energy changes when certain internal coordinates (bond lengths, angles, or torsions) are displaced.

These constants appear in the potential energy expansion:

$$
V = \tfrac{1}{2} \sum_{ij} F_{ij} \, \Delta q_i \, \Delta q_j
$$

where each $q_i$ represents an **internal coordinate** of the molecule  
(e.g., a bond stretch, bend, or torsional displacement).

---

### 📘 Table of Force Constants

| Symbol | Description | Coordinates Coupled | Typical Units |
|:-------|:-------------|:--------------------|:---------------|
| $k_{CC}$ | C=C stretch force constant | $r_{CC}$ | N·cm⁻¹ |
| $k_{CH}$ | C–H stretch force constant | $r_{CH}$ | N·cm⁻¹ |
| $k_{CCCH}$ | Coupling between C=C stretch and C–H stretch | $r_{CC}$, $r_{CH}$ | N·cm⁻¹ |
| $k_{CC\theta}$ | Coupling between C=C stretch and C–C–H bend | $r_{CC}$, $\theta$ | Å·N·cm⁻¹ |
| $k_{CH\alpha\theta}$ | Coupling between C–H stretch and C–C–H bend | $r_{CH}$, $\theta$ | Å·N·cm⁻¹ |
| $k_{\theta}$ | Mean C–C–H bend stiffness | $\theta_a$, $\theta_b$ | Å²·N·cm⁻¹ |
| $k_{\theta_a\theta_b}$ | Coupling between the two C–C–H bending coordinates | $\theta_a$, $\theta_b$ | Å²·N·cm⁻¹ |
| $k_{\tau}$ | Torsional force constant (H–C–C–H twist) | $\tau$ | Å·N·cm⁻¹ |

---

### 💬 Key Idea
- **Diagonal terms** (like $k_{CC}$ or $k_{CH}$) represent independent motions.  
- **Off-diagonal terms** (like $k_{CCCH}$ or $k_{CC\theta}$) describe *coupling* between motions.  
- These couplings explain why vibrational modes are often *mixed* rather than purely stretching or bending.

> 💡 *Later in the analysis*, these constants will determine how energy is shared among vibrations through the F and G matrices.


## 🧮 Constructing the F Matrix

The **F matrix** (force constant matrix) represents how the potential energy of the molecule changes 
with respect to the **internal coordinates** (bond stretches, bends, torsions, and couplings).  

From the *Numerical Calculation* section of your lab handout, the F matrix elements are given by:

$$
\begin{aligned}
F_{11} &= k_{CC}, & F_{12} &= \sqrt{2}\,k_{CCCH}, & F_{13} &= \sqrt{2}\,k_{CCth}, \\
F_{22} &= k_{CH}, & F_{23} &= k_{CHatha}, & F_{33} &= k_{\theta} + k_{thathb}, \\
F_{44} &= k_{CH}, & F_{45} &= k_{CHatha}, & F_{55} &= k_{\theta} - k_{thathb}, \\
F_{66} &= k_{\tau}
\end{aligned}
$$

All other matrix elements are zero.  
The matrix is **symmetric**, so $ F_{ij} = F_{ji} $.  

In this step, we’ll assemble the 6×6 matrix using the constants 
you defined earlier from `student_input`.


In [ ]:
# 🧩 Step: Build the F matrix (6×6)

import numpy as np

# ✏️ TODO 1: Create a 6×6 zero matrix
F = np.zeros((6, 6))

# --- Assign elements based on the lab handout equations ---
F[0, 0] = kCC
F[0, 1] = np.sqrt(2) * kCCCH
F[0, 2] = np.sqrt(2) * kCCth
F[1, 1] = kCH
F[1, 2] = kCHatha
F[2, 2] = ktheta + kthathb
F[3, 3] = kCH
F[3, 4] = kCHatha
F[4, 4] = ktheta - kthathb
F[5, 5] = ktau

# --- Make symmetric ---
F = F + F.T - np.diag(np.diag(F))

print("✅ F matrix constructed (N/cm):")
print(np.round(F, 4))


## ⚙️ Constructing the G Matrix (Full Expressions)

The **G matrix** represents the kinetic coupling between the six internal coordinates
shown in your Figure 2.

From the *Numerical Calculation* section of the lab handout, the matrix elements are:

$$
\begin{aligned}
A &= (\mu_H + \mu_C) \, \rho_{CH}^2, \\
B &= \mu_C \, \rho_{CC} \, (2\rho_{CC} + \rho_{CH}), \\
C &= \mu_C \, \rho_{CC} \left[ \tfrac{\rho_{CC}}{4}(1 + \cos\tau) + \rho_{CH} \right]
\end{aligned}
$$

and the $G$ elements are:

$$
\begin{aligned}
G_{11} &= 2\mu_C, \\
G_{12} &= -\sqrt{\tfrac{1}{2}} \mu_C, \\
G_{13} &= -\sqrt{\tfrac{3}{2}} \rho_{CH}\mu_C, \\
G_{22} &= \mu_C + \mu_H, \\
G_{23} &= -\sqrt{3}\rho_{CC}\mu_C \tfrac{(1 - \cos\tau)}{2}, \\
G_{33} &= A + B(1 - \cos\tau), \\
G_{44} &= G_{22}, \\
G_{45} &= -\sqrt{3}\rho_{CC}\mu_C \tfrac{(1 + \cos\tau)}{2}, \\
G_{55} &= A + B(1 + \cos\tau), \\
G_{66} &= \tfrac{8}{3}\left[A + C(1 + \cos\tau)\right].
\end{aligned}
$$

All other $G_{ij}$ values are zero in this simplified model.

---

> 💡 *Tip:* The diagonal terms represent self-coupling of internal coordinates,  
> while the off-diagonal terms ($G_{12}$, $G_{13}$, etc.) capture kinetic coupling between motions.


In [ ]:
# 🧩 Step: Build the full 6×6 G matrix based on the lab handout equations

import numpy as np

# --- Precompute constants ---
A = (muH + muC) * (rhoCH ** 2)
B = muC * rhoCC * (2 * rhoCC + rhoCH)
C = muC * rhoCC * (rhoCC * (1 + np.cos(tau)) / 4 + rhoCH)

# --- Initialize G matrix ---
G = np.zeros((6, 6))

# --- Assign elements ---
G[0, 0] = 2 * muC
G[0, 1] = -np.sqrt(1/2) * muC
G[0, 2] = -np.sqrt(3/2) * rhoCH * muC
G[1, 1] = muC + muH
G[1, 2] = -np.sqrt(3) * rhoCC * muC * (1 - np.cos(tau)) / 2
G[2, 2] = A + B * (1 - np.cos(tau))
G[3, 3] = G[1, 1]
G[3, 4] = -np.sqrt(3) * rhoCC * muC * (1 + np.cos(tau)) / 2
G[4, 4] = A + B * (1 + np.cos(tau))
G[5, 5] = (8 / 3) * (A + C * (1 + np.cos(tau)))

# --- Enforce symmetry ---
G = G + G.T - np.diag(np.diag(G))

print("✅ G matrix constructed (Å²·amu⁻¹):")
print(np.round(G, 6))


Next you will compute elements of the ${\bf G}$ matrix.  Equations for each will be given along with a code block to be completed by you.

For your reference, the variable names have been stored in python as follows:

| Equation      | Python    |
| ----------- | ----------- |
| $\mu_C$     | `muC`       |
| $\mu_H$     | `muH`       |
| $\alpha_0$  | `alpha`     |
| $\tau_0$    | `tau`       |
| $\rho_{CC}$ | `rhoCC`     |
| $\rho_{CH}$ | `rhoCH`     |




Next you will define the G matrix elements given in Eq. A9 using appropriate python syntax, for example

\begin{equation}
G_{23} = -\rho_{CC} \mu_C {\rm sin}(\alpha_0) (1 - {\rm cos}(\tau_0))
\end{equation}
would be evaluated in python as

`G23 = -rhoCC * muC * np.sin(alpha) * (1-np.cos(tau))`

Go ahead and practice typing the syntax yourself with $G_{23}$ in the block below.

In [ ]:
# MODIFY THIS BLOCK BEFORE RUNNING IT!  IT NEEDS YOUR CODING SKILLS!
G23 = -rhoCC * muC * np.sin(alpha) * (1-np.cos(tau)) # <== INSERT PYTHON CODE TO COMPUTE G23 HERE! ==>

Now try some more elements without being given the explicit python syntax. You will find a text block with the math for each element that you will compute, along with a code block just below the equation for you to complete.


$$ G_{11} = 2\mu_C $$

In [ ]:
# MODIFY THIS BLOCK BEFORE RUNNING IT!  IT NEEDS YOUR CODING SKILLS!
G11 = 2 * muC # <== INSERT PYTHON CODE TO COMPUTE G11 HERE! ==>

$$ G_{12} = \sqrt{2} \mu_C {\rm cos}(\alpha_0) $$

In [ ]:
# MODIFY THIS BLOCK BEFORE RUNNING IT!  IT NEEDS YOUR CODING SKILLS!
G12 = np.sqrt(2) * muC * np.cos(alpha) # <== INSERT PYTHON CODE TO COMPUTE G12 HERE! ==>

$$ G_{13} = -\sqrt{2} \rho_{CH} \mu_C {\rm sin}(\alpha_0) $$

In [ ]:
# MODIFY THIS BLOCK BEFORE RUNNING IT!  IT NEEDS YOUR CODING SKILLS!
G13 = -np.sqrt(2) * rhoCH * muC * np.sin(alpha) # <== INSERT PYTHON CODE TO COMPUTE G13 HERE! ==>

$$ G_{22} = G_{44} = \mu_C + \mu_H $$

In [ ]:
# MODIFY THIS BLOCK BEFORE RUNNING IT!  IT NEEDS YOUR CODING SKILLS!
G22 = muC + muH # <== INSERT PYTHON CODE TO COMPUTE G22 HERE!  ==>

# You get G44 for free!
G44 = G22

$$ G_{45} = -\rho_{CC} \mu_C  {\rm sin}(\alpha_0)(1 + {\rm cos}(\tau_0)) $$

In [ ]:
# MODIFY THIS BLOCK BEFORE RUNNING IT!  IT NEEDS YOUR CODING SKILLS!
G45 = -rhoCC * muC * np.sin(alpha) * (1 + np.cos(tau))# <== INSERT PYTHON CODE TO COMPUTE G45 HERE!  ==>

Hopefully this gives you some better feel for the terms that contribute to these elements!  To check your syntax, run the block below that will check your values.  You will see a warning that tells you which (if any) values failed this test.  If you find that one or more values failed, go back the specific lines of code where those values were computed and double check your code against the equations!

In [ ]:
# DO NOT MODIFY THIS BLOCK!!! JUST RUN IT!
expected_G11 =  2 * muC
expected_G12 = np.sqrt(2) * muC * np.cos(alpha)
expected_G13 = -np.sqrt(2) * rhoCH * muC * np.sin(alpha)
expected_G22 = muC + muH
expected_G23 = -rhoCC * muC * np.sin(alpha) * (1 - np.cos(tau))
expected_G33 = 2 * rhoCC * muC * (rhoCC - rhoCH * np.cos(alpha)) + rhoCH**2 * (muC + muH)
expected_G44 = muH + muC
expected_G45 = -rhoCC * muC * np.sin(alpha) * (1 + np.cos(tau))
expected_G55 = 2 * rhoCC * muC * (rhoCC - rhoCH * np.cos(alpha)) * (1 + np.cos(tau)) + rhoCH**2 * (muC + muH)
expected_G66 = (
    2
    / np.sin(alpha) ** 2
    * (
        2
        * rhoCC
        * muC
        * np.cos(alpha)
        * (rhoCC * np.cos(alpha) - rhoCH)
        * (1 + np.cos(tau))
        + rhoCH**2 * (muC + muH)
    )
)

if np.isclose(expected_G11, G11):
  print("G11 Passed")
else:
  print("**** G11 Failed *****")
if np.isclose(expected_G12, G12):
  print("G12 Passed")
else:
  print("**** G12 Failed *****")
if np.isclose(expected_G13, G13):
  print("G13 Passed")
else:
  print("**** G13 Failed *****")
if np.isclose(expected_G22, G22):
  print("G22 Passed")
else:
  print("**** G22 Failed *****")
if np.isclose(expected_G23, G23):
  print("G23 Passed")
else:
  print("**** G23 Failed *****")
if np.isclose(expected_G44, G44):
  print("G44 Passed")
else:
  print("**** G44 Failed *****")
if np.isclose(expected_G45, G45):
  print("G45 Passed")
else:
  print("**** G45 Failed *****")


G11 Passed
G12 Passed
G13 Passed
G22 Passed
G23 Passed
G44 Passed
G45 Passed


### PRE-WRITTEN BLOCK: Assignment of elements of ${\bf G}$ and ${\bf F}$ matrices happens here!  Do not modify this block.

In [ ]:
# DO NOT MODIFY THIS BLOCK!!! JUST RUN IT!
# initialize F and G matrices
G = np.zeros((6, 6))
F = np.zeros((6, 6))

# Assign elements of G
# G11
G[0, 0] = G11
# G12
G[0, 1] = G[1, 0] = G12
# G13
G[0, 2] = G[2, 0] =  G13
# G22
G[1, 1] = G22
# G23
G[1, 2] = G[2, 1] = G23
# G33
G[2, 2] = expected_G33
# G44
G[3, 3] = G44
# G45
G[3, 4] = G[4, 3] = G45
# G55
G[4, 4] = expected_G55
# G66
G[5, 5] = expected_G66


# Assign elements of F
F[0, 0] = kCC
F[1, 1] = kCH
F[2, 2] = kalpha
F[3, 3] = kCH
F[4, 4] = kalpha
F[5, 5] = ktau

# The vibrational frequencies are eigenvalues of ${\bf FG}$
We have now build the ${\bf G}$ and ${\bf F}$ matrices separately.  The vibrational frequencies for your system can be calculated by taking the eigenvalues of the matrix defined as the matrix product of ${\bf G}$ and ${\bf F}$.  

In python, you can compute the product of two matrices using the `@` symbol, and you can find the eigenvalues and eigenvectors using the `eig` function of the `numpy` library.  We will give you the syntax here, but you need to type this yourself in the following code block!

To form the GF matrix, type

`GF = G @ F`

To find the eigenvalues and eigenvectors (stored as variables `lam` and `v`, respectively), type

`lam, v = np.linalg.eig(GF)`

In [ ]:
# MODIFY THIS BLOCK BEFORE RUNNING IT!  IT NEEDS YOUR CODING SKILLS!

# Form the matrix product of G and F, which we will call GF
GF = G @ F

# find the eigenvalues (lam) and eigenvectors (v) of the GF matrix
lam, v = np.linalg.eig(GF)

We now have the vibrational frequencies stored in `lam`!  Yay!!!!

The last block, which has also been pre-written for you, will just print out this information with some formatting for you to use in your lab report.

In [ ]:
# DO NOT MODIFY THIS BLOCK!!! JUST RUN IT!
# sort the frequencies from highest to lowest
idx = lam.argsort()[::-1]
lam = lam[idx]
v = v[:,idx]

# convert into wavenumbers in inverse centimeters
nu_tilde = (lam * 1697200) ** 0.5



# print with formatting!
for i in range(len(v)):
  print(F'Mode {i+1}:    {nu_tilde[i]:12.3f} (cm^-1)')

# values from Mike's mathematica code - these probably won't match because
# the students will have different geometries, etc, so we won't compare them to anything!
expected_values = np.array([3101.02, 3089.5, 1551.2, 1200.43, 1174.63, 898.699])
print(kCC)

Mode 1:        3103.906 (cm^-1)
Mode 2:        3089.497 (cm^-1)
Mode 3:        1540.716 (cm^-1)
Mode 4:        1200.431 (cm^-1)
Mode 5:        1187.602 (cm^-1)
Mode 6:         962.646 (cm^-1)
7.7


**NOTE:  I don't think we need any of the text below but I'm leaving it here anyway!**

where the first three force constants are the values calculated by hand using Eqs. 3 and 4 and k_CC is an initial guess (try 7.5). Next define the F and G matrices enclosing each row in parentheses separated by commas, with an outer set of parentheses:

	F = {{kCC,0,0,0,0,0},{0,kCH,0,0,0,0},{0,0,kalpha,0,0,0},{0,0,0,kCH,0,0}.….};
	G = {{G12,G13,G14,0,0,0},{G12,G22,G23,0,0,0},{G13,G23,G33,0,0,0},….};
Perhaps the simplest way to solve the characteristic equation A10 is as an eigenvalue equation with the solutions, the lambda values, as the eigenvalues of the G×F matrix. A statement that would calculate the eigenvalues in Mathematica would look like:

	lambda = Eigenvalues[G.F];		(Matrices are multiplied using a period)

The variable lambda is a list of 6 eigenvalues in units of Ncm-1amu-1. Use Eq. 1 to calculate the vibrational wavenumbers in a subsequent statement, but this time don’t end the line with a semicolon so that the list will be displayed. When the program is run using the menu item Evaluate  Evaluate Notebook, the wavenumbers will be ordered from highest to lowest rather than in the order S_1 - S_6. One way to help identify the mode corresponding to each wavenumber is to look at the symmetry species of the mode by adding the following statements:

	ev = Eigenvectors[G.F];
	MatrixForm[ev]

This will display a matrix in which the rows contain the coefficients of the S_i for each normal mode Q_i ordered the same way as the wavenumbers. Say, for example, row 3 has all elements zero except the sixth. Then the third largest wavenumber must be associated with the out-of-plane bending mode S_6 having symmetry Ag or A1 symmetry. On the other hand, if row 3 has three nonzero elements, the third largest wavenumber must be S_1, S_2, or S_3.


